# 01 – Data inladen (maandag)
**Case 3: Mobility – Zorgt regen ervoor dat de Bikeshare minder gebruikt wordt?**

Dit notebook:
1. leest de weerdata in en bepaalt **welke periode** we gebruiken (overlap met bikeshare-data ná 2021);
2. downloadt de TfL `usage-stats`-bestanden voor die periode;
3. zet het oude én nieuwe TfL-formaat om naar één uniform formaat;
4. haalt de coördinaten van alle fietsstations op;
5. doet een eerste inspectie.

**Voorbereiding:** download de weer-CSV van Brightspace. Het notebook zoekt hem zelf op in de map van dit
notebook, in *Downloads* en in *Documenten/HVA*. Packages en mappen regelt het notebook ook.

*Conventie: code en kolomnamen in het Engels, uitleg in het Nederlands.*

## 0. Packages installeren (eenmalig, duurt een paar minuten)

In [ ]:
# Install the required packages. Only needed ONCE per computer:
# remove the '#' in front of %pip, run the cell, then put the '#' back.
# %pip install -q pandas pyarrow numpy requests matplotlib scipy statsmodels holidays streamlit plotly

In [ ]:
# Load libraries and create the folder structure (raw = untouched source data, processed = our own output)
import os, sys
from pathlib import Path
# The notebooks live in notebooks/: work from the project root so data/ and weather_impact.py are found
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import re, json, html, shutil
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)

DATA = Path("data")
RAW_TFL = DATA / "raw" / "tfl"
RAW_WEATHER = DATA / "raw" / "weather"
PROC = DATA / "processed"
for p in [RAW_TFL, RAW_WEATHER, PROC]:
    p.mkdir(parents=True, exist_ok=True)

## 1. Weerdata inspecteren
We kijken eerst naar de weerdata, omdat die bepaalt welke periode we kunnen analyseren.

In [ ]:
# Find the weather CSV (copy it into data/raw/weather if needed) and read it
def is_weather_csv(f):
    # A real weather file has a precipitation column; our own logs (weather_log.csv) don't
    if "processed" in f.parts or "log" in f.stem.lower():
        return False
    try:
        cols = [c.lower() for c in pd.read_csv(f, nrows=0).columns]
    except Exception:
        return False
    return any(k in c for c in cols for k in ["prcp", "precip", "rain", "neerslag"])

weather_files = [f for f in sorted(RAW_WEATHER.glob("*.csv")) if is_weather_csv(f)]
if not weather_files:
    # Nothing usable in data/raw/weather yet → look for a weather CSV elsewhere
    search_dirs = [Path.cwd(), Path.home() / "Documents" / "HVA", Path.home() / "Downloads"]
    candidates = []
    for d in search_dirs:
        if d.exists():
            candidates += [f for f in d.rglob("*.csv")
                           if re.search(r"weather|weer", f.name, re.I) and is_weather_csv(f)]
    # weather_london.csv first, then the most recent file
    candidates = sorted(set(candidates), key=lambda f: (f.name.lower() != "weather_london.csv", -f.stat().st_mtime))
    if candidates:
        shutil.copy(candidates[0], RAW_WEATHER / candidates[0].name)
        print("Weer-CSV gekopieerd van:", candidates[0])
    weather_files = [f for f in sorted(RAW_WEATHER.glob("*.csv")) if is_weather_csv(f)]
assert weather_files, ("Geen weer-CSV gevonden. Zet weather_london.csv in data/raw/weather/ en draai deze cel opnieuw.")
# weather_london.csv gets priority if there are several files
weather_files = sorted(weather_files, key=lambda f: f.name.lower() != "weather_london.csv")
WEATHER_FILE = weather_files[0]
weather_raw = pd.read_csv(WEATHER_FILE)
print(WEATHER_FILE, weather_raw.shape)
weather_raw.head()

In [ ]:
# Column types and number of non-empty values per column
weather_raw.info()

In [ ]:
# Summary statistics (min/max/mean/quartiles) to spot impossible values
weather_raw.describe().T

In [ ]:
# Missing values per column (count and %)
missing = weather_raw.isna().sum().to_frame("n_missing")
missing["pct_missing"] = (missing["n_missing"] / len(weather_raw) * 100).round(2)
missing

### Kolommen herkennen
De functie hieronder raadt welke kolom de datum, neerslag en temperatuur bevat.
**Controleer de output!** Klopt het niet, vul de kolomnaam dan zelf in.

In [ ]:
# Guess which columns hold the date, precipitation and temperature (check the printed result!)
def find_column(df, candidates):
    for k in candidates:
        for c in df.columns:
            if k in c.lower():
                return c
    return None

DATE_COL = find_column(weather_raw, ["date", "datum", "time", "day", "unnamed"])
if DATE_COL is None:
    # No recognisable name: use the first column that can be read as dates
    for c in weather_raw.columns:
        if pd.to_datetime(weather_raw[c].astype(str), errors="coerce").notna().mean() > 0.9:
            DATE_COL = c
            break
assert DATE_COL is not None, f"Geen datumkolom gevonden in {WEATHER_FILE}. Kolommen: {list(weather_raw.columns)}"
RAIN_COL = find_column(weather_raw, ["precip", "rain", "prcp", "neerslag"])
TEMP_COL = find_column(weather_raw, ["mean_temp", "tavg", "temp_mean", "avg_temp", "temp"])

# Override manually if needed, e.g.: RAIN_COL = "precipitation"
print("Datum:", DATE_COL, "| Neerslag:", RAIN_COL, "| Temperatuur:", TEMP_COL)

In [ ]:
# Convert the date column from text/number to a real datetime
def parse_date(s):
    # Handles 20230115 (number), '2023-01-15' and '15/01/2023'
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_datetime(s.astype("Int64").astype(str), format="%Y%m%d", errors="coerce")
    first = str(s.dropna().iloc[0])
    dayfirst = bool(re.match(r"^\d{1,2}[/-]\d{1,2}[/-]\d{4}", first))
    return pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)

weather_raw["date"] = parse_date(weather_raw[DATE_COL]).dt.normalize()
print("Niet te parsen datums:", weather_raw["date"].isna().sum())
print("Weerdata loopt van", weather_raw["date"].min().date(), "t/m", weather_raw["date"].max().date())

## 2. Periode kiezen
De bikeshare-data moet van **ná 2021** zijn (advies docent). We kiezen het **meest recente volledige
kalenderjaar** dat in de weerdata zit. Een volledig jaar is belangrijk: dan zitten alle seizoenen erin en
kunnen we regen eerlijk vergelijken *binnen* dezelfde maand.

Als er geen volledig jaar overlapt, pakken we de hele overlap. Lukt dat ook niet → zie de melding.

In [ ]:
# Pick the analysis period: the most recent full calendar year covered by both weather and bike data (after 2021)
w_min, w_max = weather_raw["date"].min(), weather_raw["date"].max()
BIKE_MIN = pd.Timestamp("2021-01-01")

overlap_start = max(w_min, BIKE_MIN)
if overlap_start > w_max:
    raise ValueError(
        f"Geen overlap! Weerdata stopt op {w_max.date()}, bikeshare-data moet na 2021 zijn. "
        "Oplossing: voeg zelf recentere weerdata toe (bijv. via Meteostat / Open-Meteo, mag volgens de opdracht)."
    )

full_years = [y for y in range(overlap_start.year, w_max.year + 1)
              if pd.Timestamp(f"{y}-01-01") >= overlap_start and pd.Timestamp(f"{y}-12-31") <= w_max]
if full_years:
    YEAR = max(full_years)
    START, END = pd.Timestamp(f"{YEAR}-01-01"), pd.Timestamp(f"{YEAR}-12-31")
else:
    START, END = overlap_start, w_max

# Manual override allowed, e.g.: START, END = pd.Timestamp("2022-01-01"), pd.Timestamp("2022-12-31")
print(f"Gekozen periode: {START.date()} t/m {END.date()}  ({(END - START).days + 1} dagen)")

In [ ]:
# Save the choices so notebook 02 uses exactly the same settings
config = {
    "start": str(START.date()), "end": str(END.date()),
    "weather_file": str(WEATHER_FILE),
    "date_col": DATE_COL, "rain_col": RAIN_COL, "temp_col": TEMP_COL,
}
(PROC / "config.json").write_text(json.dumps(config, indent=2))
config

## 3. TfL-bestanden zoeken en downloaden
TfL zet elke week een bestand online (bijv. `296JourneyDataExtract03Jan2022-09Jan2022.csv`).
We vragen de lijst met bestanden op en downloaden alleen de weken die in onze periode vallen.

⏱️ Een heel jaar is ±52 bestanden en 1–2 GB. Reken op 5–15 minuten. Al gedownloade bestanden worden overgeslagen,
dus je kunt de cel gerust opnieuw draaien als hij halverwege stopt.

In [ ]:
# Get the list of all TfL files, read the week dates from each file name and keep only weeks in our period
BUCKET = "https://s3-eu-west-1.amazonaws.com/cycling.data.tfl.gov.uk"

def list_tfl_files(prefix="usage-stats/"):
    keys, token = [], None
    while True:
        params = {"list-type": "2", "prefix": prefix}
        if token:
            params["continuation-token"] = token
        r = requests.get(BUCKET + "/", params=params, timeout=30)
        r.raise_for_status()
        keys += [html.unescape(k) for k in re.findall(r"<Key>(.*?)</Key>", r.text)]
        m = re.search(r"<NextContinuationToken>(.*?)</NextContinuationToken>", r.text)
        if not m:
            return keys
        token = html.unescape(m.group(1))

MONTHS = dict(jan=1, feb=2, mar=3, apr=4, may=5, jun=6, jul=7, aug=8, sep=9, oct=10, nov=11, dec=12)
PATTERN = re.compile(r"(\d{1,2})([A-Za-z]{3,4})(\d{4})-(\d{1,2})([A-Za-z]{3,4})(\d{4})")

def period_from_name(name):
    m = PATTERN.search(name)
    if not m:
        return None
    d1, m1, y1, d2, m2, y2 = m.groups()
    try:
        return (pd.Timestamp(int(y1), MONTHS[m1[:3].lower()], int(d1)),
                pd.Timestamp(int(y2), MONTHS[m2[:3].lower()], int(d2)))
    except (KeyError, ValueError):
        return None

try:
    all_keys = list_tfl_files()
    print("Bestanden op de TfL-server:", len(all_keys))
except Exception as e:
    print("Kon de lijst niet ophalen:", e, "→ we gebruiken alleen bestanden die al in data/raw/tfl staan.")
    all_keys = []

needed = []
for k in all_keys:
    p = period_from_name(k)
    if k.lower().endswith(".csv") and "journeydataextract" in k.lower() and p and p[1] >= START and p[0] <= END:
        needed.append(k)
print("Bestanden in de gekozen periode:", len(needed))

In [ ]:
# Download the files we still need (files already on disk are skipped, so re-running is safe)
for i, k in enumerate(needed, 1):
    target = RAW_TFL / Path(k).name
    if target.exists() and target.stat().st_size > 0:
        continue
    print(f"[{i}/{len(needed)}] {Path(k).name}")
    r = requests.get(f"{BUCKET}/{quote(k)}", timeout=300)
    r.raise_for_status()
    target.write_bytes(r.content)

local_files = sorted(f for f in RAW_TFL.glob("*.csv")
                     if (p := period_from_name(f.name)) and p[1] >= START and p[0] <= END)
print("Klaar. Bestanden lokaal beschikbaar:", len(local_files))

## 4. Inladen in één uniform formaat
**Let op:** TfL veranderde rond september 2022 het formaat, en is ook binnen een formaat niet altijd consequent
met kolomnamen (`EndStation Id` vs. `End Station Id`, `Duration` vs. `Duration_Seconds`, ...).

| | Oud formaat | Nieuw formaat |
|---|---|---|
| Datum | `13/01/2022 10:00` | `2022-10-13 10:00` |
| Duur | `Duration` (seconden) | `Total duration (ms)` |
| Station-ID | BikePoint-ID (bijv. `1`) | Terminal-nummer (bijv. `1023`) |

Oplossing: kolomnamen eerst **normaliseren** (kleine letters, alleen letters/cijfers), dan vertalen naar één
vaste set kolomnamen. We onthouden welk type station-ID het is (`id_type`).

In [ ]:
# Read every weekly file and convert both TfL formats to one set of column names
def norm(c):
    # "End Station Id" -> "endstationid"
    return re.sub(r"[^a-z0-9]", "", str(c).lower())

COLUMN_MAP = {
    "number": "ride_id", "rentalid": "ride_id",
    "startdate": "start_time", "enddate": "end_time",
    "startstationnumber": "start_station_id", "startstationid": "start_station_id",
    "endstationnumber": "end_station_id", "endstationid": "end_station_id",
    "startstation": "start_station", "startstationname": "start_station",
    "endstation": "end_station", "endstationname": "end_station",
    "startstationlogicalterminal": "start_terminal", "endstationlogicalterminal": "end_terminal",
    "bikemodel": "bike_model",
    "totaldurationms": "duration_ms", "duration": "duration_s", "durationseconds": "duration_s",
}
REQUIRED = ["ride_id", "start_time", "end_time", "start_station_id", "start_station", "end_station_id", "end_station"]
OUTPUT_COLS = ["ride_id", "start_time", "end_time", "start_station_id", "start_station", "end_station_id",
               "end_station", "bike_model", "duration_min", "id_type", "source_file"]

def parse_time(s):
    first = str(s.dropna().iloc[0]) if s.notna().any() else ""
    if re.match(r"^\d{4}-", first):
        return pd.to_datetime(s, errors="coerce", format="ISO8601")
    return pd.to_datetime(s, errors="coerce", dayfirst=True, format="mixed")

def read_tfl(path):
    df = pd.read_csv(path, usecols=lambda c: norm(c) in COLUMN_MAP, low_memory=False)
    df.columns = [COLUMN_MAP[norm(c)] for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]

    # New format (ms) uses terminal numbers, old format uses BikePoint IDs
    id_type = "terminal" if "duration_ms" in df.columns else "bikepoint"
    # No regular station ID but a 'Logical Terminal' column? Use that instead (= terminal number)
    for side in ["start", "end"]:
        if f"{side}_station_id" not in df.columns and f"{side}_terminal" in df.columns:
            df[f"{side}_station_id"] = df[f"{side}_terminal"]
            id_type = "terminal"
    df["id_type"] = id_type

    missing_cols = [c for c in REQUIRED if c not in df.columns]
    if missing_cols:
        print(f"  ⚠️ {path.name}: kolommen {missing_cols} niet gevonden (blijven leeg). "
              f"Kolommen in dit bestand: {list(pd.read_csv(path, nrows=0).columns)}")
        for c in missing_cols:
            df[c] = pd.NA

    if "duration_ms" in df.columns:
        df["duration_min"] = pd.to_numeric(df["duration_ms"], errors="coerce") / 60_000
    elif "duration_s" in df.columns:
        df["duration_min"] = pd.to_numeric(df["duration_s"], errors="coerce") / 60
    else:
        df["duration_min"] = np.nan
    df["start_time"], df["end_time"] = parse_time(df["start_time"]), parse_time(df["end_time"])
    # No duration in the file? Compute it from start and end time
    df["duration_min"] = df["duration_min"].fillna((df["end_time"] - df["start_time"]).dt.total_seconds() / 60)

    for c in ["start_station_id", "end_station_id", "ride_id"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    if "bike_model" not in df.columns:
        df["bike_model"] = pd.NA
    df["source_file"] = path.name
    return df[OUTPUT_COLS]

# Reading ~53 large CSVs takes a few minutes. If rides_raw.parquet already exists, is newer than all
# CSVs and has the current column names, we load that instead. Set REBUILD = True to force a fresh read.
REBUILD = False
cache = PROC / "rides_raw.parquet"
use_cache = bool(not REBUILD and cache.exists() and local_files
                 and cache.stat().st_mtime > max(f.stat().st_mtime for f in local_files))

if use_cache:
    rides = pd.read_parquet(cache)
    use_cache = set(OUTPUT_COLS) <= set(rides.columns) and rides["source_file"].nunique() == len(local_files)

if use_cache:
    print(f"Bestaande {cache.name} geladen (CSV's niet opnieuw ingelezen).")
else:
    parts = []
    for f in local_files:
        d = read_tfl(f)
        parts.append(d)
        print(f"{f.name}: {len(d):,} ritten  ({d['id_type'].iloc[0] if len(d) else '-'})")
    rides = pd.concat(parts, ignore_index=True)
    for c in ["start_station", "end_station", "bike_model", "id_type", "source_file"]:
        rides[c] = rides[c].astype("category")
    rides.to_parquet(cache, index=False)
print(f"\nTotaal: {len(rides):,} ritten")

In [ ]:
# Save all rides as parquet (much smaller and faster than CSV)
rides.head()

## 5. Stationslocaties (TfL BikePoint API)
Voor de kaart hebben we coördinaten nodig. De API geeft per station:
`BikePoints_<id>` (oud formaat-ID), `TerminalName` (nieuw formaat-ID), naam, lat/lon en aantal docks.

Stations die inmiddels zijn opgeheven staan niet meer in de API. In notebook 02 kijken we hoeveel ritten we
daardoor niet kunnen koppelen.

In [ ]:
# Get the location (lat/lon), IDs and number of docks of every bike station from the TfL API
try:
    r = requests.get("https://api.tfl.gov.uk/BikePoint", timeout=60)
    r.raise_for_status()
    rows = []
    for bp in r.json():
        props = {p["key"]: p["value"] for p in bp.get("additionalProperties", [])}
        rows.append({
            "bikepoint_id": int(bp["id"].split("_")[-1]),
            "terminal_id": pd.to_numeric(props.get("TerminalName"), errors="coerce"),
            "name": bp["commonName"], "lat": bp["lat"], "lon": bp["lon"],
            "docks": pd.to_numeric(props.get("NbDocks"), errors="coerce"),
        })
    stations = pd.DataFrame(rows)
    stations.to_parquet(PROC / "stations.parquet", index=False)
except Exception as e:
    print("API niet bereikbaar:", e)
    stations = pd.read_parquet(PROC / "stations.parquet")  # previously saved version

print(len(stations), "stations")
stations.head()

In [ ]:
# Quick map check: do the stations look like London?
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(stations["lon"], stations["lat"], s=stations["docks"].fillna(10), alpha=.5)
ax.set_title("Bikeshare-stations in Londen (grootte = aantal docks)")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
plt.show()

## 6. Eerste inspectie
Snelle checks om problemen vroeg te zien. Schrijf je bevindingen op: die heb je nodig voor de
opschoon-tab in het dashboard (rubric **Opschonen**).

In [ ]:
# Health check (nothing is removed here): missing values, duplicates, suspicious durations, file formats
print("Periode in de data:", rides["start_time"].min(), "t/m", rides["start_time"].max())
print("Ontbrekende waarden:")
print(rides.isna().sum())
print("\nDubbele rit-ID's:", rides["ride_id"].duplicated().sum())
print("Ritten < 1 minuut:", (rides["duration_min"] < 1).sum())
print("Ritten > 24 uur:", (rides["duration_min"] > 24 * 60).sum())
print("Formaat per bestand:")
print(rides.groupby("id_type", observed=True)["source_file"].nunique())

In [ ]:
# Distribution of ride duration and number of rides per day (a day with 0 rides means a missing file)
rides["duration_min"].clip(upper=120).hist(bins=60, figsize=(8, 3))
plt.title("Ritduur (min, afgekapt op 120)"); plt.show()

rides_per_day = rides.set_index("start_time").resample("D").size()
rides_per_day.plot(figsize=(12, 3), title="Ritten per dag (ruw)"); plt.show()

### Bevindingen (vul zelf in)
- …
- …